# Proyecto de Ciencia de Datos & MLOps: Calidad de Vinos
### Especialización en Ciencia de Datos - Trabajo de Grado 1
**Institución:** Universidad Santo Tomás - Seccional Tunja  
**Arquitectura:** Delta Lake + MLflow Experiments + Model Registry + Model Serving (API REST)

---

## 1. Formulación del Problema y Pregunta Analítica
¿Es posible entrenar, registrar y desplegar en tiempo real un modelo de Machine Learning capaz de clasificar la calidad de un vino con alta precisión y baja latencia de inferencia?

## 2. Formulación de Hipótesis Científica
* **$H_0$ (Hipótesis Nula):** La profundidad máxima del ensamble (`max_depth`) no influye en la métrica de generalización en datos no vistos.
* **$H_1$ (Hipótesis Alternativa):** Aumentar `max_depth` mejora el rendimiento en entrenamiento, pero superado un umbral genera sobreajuste (*overfitting*), degradando las métricas en prueba.

## 3. Ingesta y Persistencia en Delta Lake (Catalog)
Convertimos los datos reales de la UCI en una tabla **Delta Lake** para garantizar gobernanza, versionamiento de datos (Time Travel) y transacciones ACID en el Catálogo de Databricks.

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine

# Carga del conjunto de datos real de vinos de la UCI
wine = load_wine(as_frame=True)
df = wine.frame
df['target'] = (wine.target == 0).astype(int)

# Persistencia en formato Delta Lake en el catálogo de Databricks
spark_df = spark.createDataFrame(df)
spark_df.write.format('delta').mode('overwrite').saveAsTable('default.calidad_vinos_uci')

print('Tabla Delta default.calidad_vinos_uci persistida exitosamente.')
display(spark.sql('SELECT * FROM default.calidad_vinos_uci LIMIT 5'))

## 4. Experimentación Científica con MLflow Tracking (Experiments)
Entrenamos variando hiperparámetros y registrando de forma automática cada ejecución con sus métricas, hiperparámetros, firma de datos y artefactos binarios.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from mlflow.models import infer_signature

# Lectura desde la tabla Delta gobernada
data = spark.table('default.calidad_vinos_uci').toPandas()
X = data.drop(columns=['target', 'target_name'], errors='ignore')
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# En Databricks, las ejecuciones se vinculan automáticamente al experimento del cuaderno.
# Opcionalmente se puede definir la ruta compartida universal:
try:
    mlflow.set_experiment('/Shared/Experimento_Vinos_MLOps')
except Exception:
    pass  # Si no hay permisos en /Shared, registra en el experimento local del cuaderno

depths = [2, 4, 8, None]
for d in depths:
    with mlflow.start_run(run_name=f'RandomForest_depth_{d}'):
        model = RandomForestClassifier(n_estimators=50, max_depth=d, random_state=42)
        model.fit(X_train, y_train)
        
        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average='weighted')
        
        # Inferencia de la firma estricta de datos
        signature = infer_signature(X_train, model.predict(X_train))
        
        # Registro de parámetros, métricas y artefactos
        mlflow.log_param('max_depth', str(d))
        mlflow.log_metric('test_accuracy', acc)
        mlflow.log_metric('test_f1', f1)
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path='modelo_rf',
            signature=signature,
            input_example=X_train.iloc[:2]
        )
        print(f'Run registrado en MLflow: max_depth={str(d):<5} | Accuracy={acc:.3f} | F1={f1:.3f}')


## 5. Registro y Versionamiento en Model Registry (Models)
Promovemos el mejor modelo del experimento al catálogo central de modelos de la empresa para habilitar linaje y gobierno.

In [ ]:
model_name = 'Clasificador_Vinos_Produccion'

# Tomamos el run óptimo y lo registramos formalmente en el Model Registry
run_id = mlflow.last_active_run().info.run_id
model_uri = f'runs:/{run_id}/modelo_rf'

registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)
print(f'Modelo registrado exitosamente en Model Registry con Versión: {registered_model.version}')